# Januário Leads — Piloto Overture GRATUITO (sem GitHub Actions)

Este notebook importa **até 30 estabelecimentos reais de um único município** para o **Supabase de homologação**, utilizando o Google Colab gratuito (CPU). A versão principal do site e o banco de produção não são alterados.

**Ordem segura:** Varginha/MG → Pouso Alegre/MG → Campinas/SP → Niterói/RJ. Execute um município por vez, confira o resultado, então escolha o próximo. O importador interrompe quando o banco atinge **100 MiB**; não é um processo nacional automático.

**Segurança:** a chave `service_role` da homologação é secreta e permite acesso amplo ao banco. Nunca cole a chave em células de código, comentários, arquivos, GitHub ou no chat. Use o ícone de chave 🔑 do Colab para configurar um segredo chamado `OVERTURE_SUPABASE_SERVICE_ROLE_KEY` e autorize este notebook a lê-lo. Se usar a alternativa `getpass`, a chave é digitada de forma oculta e permanece apenas na memória temporária da sessão. Desconecte o runtime após o uso.

**Custos:** Colab possui modalidade sem cobrança, mas com recursos/tempo limitados; o Supabase também tem limites próprios. Este piloto não habilita nenhum serviço pago e não garante cobertura completa da cidade.

## 1. Escolher o município e conferir limites

Altere apenas `UF` e `MUNICIPIO` entre os quatro pilotos autorizados. O padrão é Varginha/MG.

In [ ]:
UF = "MG"
MUNICIPIO = "Varginha"
LIMITE_REGISTROS = 30
TETO_BANCO_MIB = 100

PILOTOS = {
    "MG": {"Varginha", "Pouso Alegre"},
    "SP": {"Campinas"},
    "RJ": {"Niterói"},
}
assert UF in PILOTOS and MUNICIPIO in PILOTOS[UF], "Escolha apenas um município do piloto."
assert 1 <= LIMITE_REGISTROS <= 30
assert TETO_BANCO_MIB == 100
print(f"Piloto escolhido: {MUNICIPIO}/{UF}; máximo {LIMITE_REGISTROS} leads; teto {TETO_BANCO_MIB} MiB.")


## 2. Preparar o ambiente Colab

A primeira execução instala bibliotecas geográficas e baixa **somente o importador já aprovado** do repositório (versão fixada em um commit). Requer conexão do Colab à internet; a rede deste chat não executa esses downloads.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from urllib.request import urlretrieve

WORKDIR = Path("/content/januario_overture_piloto")
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
print("Instalando dependências geográficas no ambiente temporário...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "overturemaps==2.1.1", "geopandas>=1.0", "shapely>=2.0", "pyogrio>=0.10"
], check=True)

# Script validado e fixado no commit da PR #142, sem código editável remoto.
SCRIPT_URL = (
    "https://raw.githubusercontent.com/wesleyshoto702-hub/januarioleads/"
    "53771f80e0acebd8060afd881ac16e10e85a4a97/"
    "scripts/import_overture_br.py"
)
SCRIPT = WORKDIR / "import_overture_br.py"
urlretrieve(SCRIPT_URL, SCRIPT)
assert SCRIPT.stat().st_size > 5000, "Falha ao baixar o importador."
print("Importador do Januário Leads preparado.")


## 3. Liberar a chave da **homologação** com segurança

No Colab, abra 🔑 **Segredos** e cadastre `OVERTURE_SUPABASE_SERVICE_ROLE_KEY`. Ative o acesso do notebook. Se não houver segredo disponível, será solicitado um valor oculto. **NÃO use a chave de produção.**

In [ ]:
from getpass import getpass

SUPABASE_URL = "https://prlrxjothrslhswpkoza.supabase.co"  # STAGING, fixo
try:
    from google.colab import userdata
    private_key = userdata.get("OVERTURE_SUPABASE_SERVICE_ROLE_KEY")
except Exception:
    private_key = None

if not private_key:
    private_key = getpass("Cole somente a chave service_role da HOMOLOGAÇÃO (entrada oculta): ")

if not private_key or len(private_key.strip()) < 20:
    raise ValueError("A chave de homologação está ausente ou inválida.")

# Não imprimir a chave, nem salvar em arquivo/Drive/notebook.
os.environ["SUPABASE_URL"] = SUPABASE_URL
os.environ["SUPABASE_SERVICE_ROLE_KEY"] = private_key.strip()
del private_key
print("Credencial temporária preparada para a homologação. Valor oculto.")


## 4. Importar estabelecimentos reais

Este passo acessa IBGE + Overture, faz o recorte municipal e salva no Supabase **de homologação**. Pode levar algum tempo ou ser interrompido pelo Colab gratuito. Se falhar, leia o erro; não tente ampliar os limites.

In [ ]:
assert SUPABASE_URL == "https://prlrxjothrslhswpkoza.supabase.co"
assert SCRIPT.is_file()
cmd = [
    sys.executable, str(SCRIPT),
    "--states", UF,
    "--cities", MUNICIPIO,
    "--max-records-per-state", str(LIMITE_REGISTROS),
    "--max-records-per-city", str(LIMITE_REGISTROS),
    "--database-max-mb", str(TETO_BANCO_MIB),
]
# Nunca passar chaves ou segredos como argumentos imprimíveis.
print(f"Importando até {LIMITE_REGISTROS} estabelecimentos de {MUNICIPIO}/{UF}...")
completed = subprocess.run(cmd, cwd=str(WORKDIR), check=False)
if completed.returncode != 0:
    raise RuntimeError(f"Importação interrompida (código {completed.returncode}). "
                       "Verifique a mensagem acima antes de qualquer nova tentativa.")
print("Importador terminou; confira a contagem real na próxima célula.")


## 5. Verificar a quantidade importada

Consulta apenas o total de registros da cidade, sem mostrar credenciais. Se aparecer zero, **não declare importação concluída**. Depois do piloto, desconecte o ambiente de execução.

In [ ]:
import urllib.request
from urllib.parse import urlencode

params = urlencode({
    "select": "id",
    "state": f"eq.{UF}",
    "city_key": "eq." + __import__("unicodedata").normalize("NFD", MUNICIPIO.lower()).encode("ascii", "ignore").decode("ascii").replace("-", " "),
    "limit": "1",
})
req = urllib.request.Request(
    SUPABASE_URL + "/rest/v1/overture_places?" + params,
    headers={
        "apikey": os.environ["SUPABASE_SERVICE_ROLE_KEY"],
        "Authorization": "Bearer " + os.environ["SUPABASE_SERVICE_ROLE_KEY"],
        "Prefer": "count=exact",
    },
)
with urllib.request.urlopen(req, timeout=25) as response:
    total = response.headers.get("Content-Range", "indisponível")
print(f"Leads Overture para {MUNICIPIO}/{UF} — Content-Range: {total}")
print("Não esqueça: piloto de uma cidade ≠ cobertura completa da UF/Brasil.")
